# Choosing the search settings, tested on my questions (KAN-221, first look for KAN-99)
Gözde Nur Özdemir · ELIZA · Sprint 2

**What this notebook does:** it tests the choices behind the search (the embedding model is checked in `embedding.ipynb`) in `src/embedding.ipynb` and writes the winners in `config/rag.yaml`. Every choice is either taken from a source or measured here.

| Step | Question | Where the choice comes from | Jira |
|---|---|---|---|
| 1 | Which questions, and how do we score? | instructor's slides: *"compute Recall@k / MRR on gold IDs"* | |
| 2 | Should words be removed at all? | Dickinson 2026, *Stop Removing Stopwords* | KAN-221 |
| 3 | Which words does BM25 already ignore? | the BM25 formula (IDF) | KAN-221 |
| 4 | Which frequent words are useless? A jury of three LLMs | instructor's feedback; Verga et al. 2024, *Replacing Judges with Juries* | KAN-221 |
| 5 | Does removing them help? | Dickinson 2026: *"the burden of proof sits with removal"* | KAN-221 |
| 6 | Dense, BM25 or hybrid? | design decision D12 | KAN-99 (first look) |
| 7 | How many chunks (top-k)? | design decision D12 | KAN-99 (first look) |

**Questions used:** only my questions on Annex III 3(b), **B-01 to B-06**. The team's other questions are kept for the `val`/`test` split (KAN-53) and the full measurement in KAN-99.

**Install once, in the terminal** (not in the notebook), in the same Python environment as Jupyter:
```
pip install sentence-transformers chromadb rank_bm25 pandas pyyaml requests scikit-learn umap-learn matplotlib
```

## 1. Which questions, and how do we score?
For each of B-01 to B-06 the team wrote the **gold source**: the provisions that answer it.
- **Recall@k:** share of the gold provisions in the top k chunks. A question with 4 gold provisions and 2 of them in the top 5 counts as 0.5, not as a success.
- **MRR:** 1 / rank of the first correct chunk (1.0 = always first).
- **Precision@k:** share of the top k chunks that are gold provisions; unrelated chunks lower it.

In [ ]:
import csv, json, os, re
from collections import Counter
import numpy as np, pandas as pd, requests, yaml
pd.set_option("display.max_colwidth", None)
np.random.seed(42)

IN_REPO = os.path.exists("../config/rag.yaml")      # opened from src/ in the repository
FOLDER = os.path.expanduser("~/ELIZA")              # otherwise: the folder "ELIZA" in your home folder, with all files
P = lambda path: path if IN_REPO else os.path.join(FOLDER, os.path.basename(path))
CONFIG = yaml.safe_load(open(P("../config/rag.yaml")))
chunks = [json.loads(line) for line in open(P("../data/chunks.jsonl"), encoding="utf-8")]
questions = {r["id"]: r for r in csv.DictReader(open(P("../docs/eval/eval-set-v1.csv"), encoding="utf-8-sig"))}
ref_of = {c["chunk_id"]: c["ref"] for c in chunks}
words = lambda text: re.findall(r"[a-z0-9]+", text.lower())
print(len(chunks), "chunks,", len(questions), "questions")
dev = [questions[q] for q in ["B-01", "B-02", "B-03", "B-04", "B-05", "B-06"]]

def score(search, k=5):
    """Recall@k, MRR and Precision@k over my questions B-01 to B-06."""
    recall, rr, precision = [], [], []
    for q in dev:
        gold = [g.strip() for g in q["gold_source"].split(";")]
        refs = [ref_of[c] for c in search(q["question"])[:k]]
        hit = lambda r: any(r.startswith(g) or g.startswith(r) for g in gold)
        recall.append(np.mean([any(r.startswith(g) or g.startswith(r) for r in refs) for g in gold]))
        rr.append(next((1 / n for n, r in enumerate(refs, 1) if hit(r)), 0))
        precision.append(np.mean([hit(r) for r in refs]))
    return {"Recall@5" if k == 5 else f"Recall@{k}": round(np.mean(recall), 2), "MRR": round(np.mean(rr), 2),
            "Precision@5" if k == 5 else f"Precision@{k}": round(np.mean(precision), 2)}

pd.DataFrame(dev)[["id", "question", "gold_source"]]

## 2. Should words be removed at all?
**Dense search (embedding)** reads the full `embed_text`: heading + the sentence before a point + the provision. The model reads normal sentences, and small words change a rule: *"shall **not** be considered to be high-risk"* means the opposite without "not".

**Keyword search (BM25)** reads the same text in lower case, split into words. We **do not remove stopwords by default**. A 2026 study on court decisions (Dickinson, *Stop Removing Stopwords*) found that the usual stopword lists (scikit-learn, NLTK, spaCy) made legal text classification slightly worse in all 12 tests, and that even word-by-word tuned lists gave no measurable gain. Its rule: *"the burden of proof sits with removal"*: a word is removed only if removing it is shown to help (step 5).

In [ ]:
example = next(c for c in chunks if c["chunk_id"] == "ART6-3-1")
print("TEXT:", example["text"][:150], "...")
print("BM25:", words(example["text"])[:18], "...")
print(sum(any(w in {"not", "no", "nor"} for w in words(c["text"])) for c in chunks), "of 105 chunks contain a negation")

**Comment:** Nothing is removed yet: BM25 sees “by”, “from” and “an”, and also “not”. 27 of the 105 chunks contain a negation, so a careless stopword list could change the meaning of about a quarter of the provisions.

## 3. Which words does BM25 already ignore?
BM25 gives every word a weight (IDF): a word in many chunks gets a weight close to 0, so it hardly counts in a search. For comparison, a word in only one chunk gets about 4.3. This is part of BM25 itself, so frequent words are already pushed down without any list. The table shows the **20 words with the lowest weight**, how many chunks and questions contain them. These 20 are only the candidates for step 4.

In [ ]:
from rank_bm25 import BM25Okapi
in_q = Counter(w for q in questions.values() for w in set(words(q["question"])))
in_c = Counter(w for c in chunks for w in set(words(c["embed_text"])))

N = len(chunks)
idf = lambda n: np.log(1 + (N - n + 0.5) / (n + 0.5))     # BM25 word weight (as in Lucene / Elasticsearch)
weights = pd.DataFrame([{"word": w, "BM25 weight (IDF)": round(idf(n), 2), "% of chunks": round(100 * n / N),
                         "% of questions": round(100 * in_q[w] / len(questions))} for w, n in in_c.items()])
candidates = weights.sort_values("BM25 weight (IDF)").head(20).reset_index(drop=True)
candidates

**Comment:** Frequent words already get very low weights: “article” is in every chunk (weight 0.00) and “the” in 91% (0.09), while a word in only one chunk gets about 4.3. But some low-weight words are in many questions (“ai” 73%, “risk” and “high” 46%), so frequency alone cannot decide what to drop.

## 4. A jury of three LLMs labels the candidates
The instructor asked us to let an LLM decide which frequent words are useless, with a **different model** than our answer model. Instead of one judge, we use a **jury of three small models from different companies**, and the majority wins. Verga et al. (2024, *Replacing Judges with Juries*) showed that a panel of three models from different families agrees better with people than one large judge (Cohen's kappa 0.76 vs 0.63 for GPT-4 on Natural Questions), has less bias towards its own family, and is cheaper; for yes/no labels they use the majority vote.

| Model | Company | Why |
|---|---|---|
| `qwen3:4b` | Alibaba | worked with thinking turned off in our model selection spike |
| `phi4-mini` | Microsoft | our fallback answer model in the spike (D9) |
| `gemma3:4b` | Google | a third family of the same size |

None is **Llama** (our answer model, D9) or **Claude** (used to draft the questions). Same prompt, three examples of each word from our chunks, temperature 0. Needs `ollama pull qwen3:4b`, `ollama pull phi4-mini`, `ollama pull gemma3:4b`.

In [ ]:
PROMPT = """This word appears in many EU AI Act provisions. Examples:
{examples}
Should a keyword search IGNORE it (common wording that does not help to find the right provision)
or KEEP it (a question can depend on it)? If in doubt, KEEP.
Answer in one line: IGNORE or KEEP, then a reason of at most 12 words.
WORD: {word}"""

installed = [m["name"] for m in requests.get(CONFIG["judge"]["url"].replace("/api/chat", "/api/tags")).json()["models"]]
missing = [m for m in CONFIG["judge"]["jury"] if not any(i.startswith(m) for i in installed)]
assert not missing, f"Run in the terminal first: {'  '.join('ollama pull ' + m for m in missing)}"

def ask(model, prompt):
    """The final answer of a model in Ollama: temperature 0, thinking off for Qwen3."""
    body = {"model": model, "stream": False, "messages": [{"role": "user", "content": prompt}],
            "options": {"temperature": 0, "num_predict": 2000}}     # room to finish thinking if Ollama cannot switch it off
    if model.startswith("qwen3"):
        body["think"] = False
        body["messages"][0]["content"] += " /no_think"         # Qwen3's own switch, also works with older Ollama
    answer = requests.post(CONFIG["judge"]["url"], json=body, timeout=300).json()
    if "error" in answer:
        raise RuntimeError(f"Ollama, {model}: {answer['error']}")
    text = answer["message"]["content"]
    if model.startswith("qwen3") and "</think>" not in text and text.lstrip().lower().startswith(("okay", "we are", "hmm")):
        return "UNCLEAR (thinking did not finish)"
    return text.split("</think>")[-1].strip()                  # keep only the final answer, after any thinking

def label(text):
    """IGNORE or KEEP if the final answer starts with it, otherwise UNCLEAR."""
    m = re.match(r"\W*(IGNORE|KEEP)\b", text.upper())
    return m[1] if m else "UNCLEAR"

def examples(word):
    return "\n".join([f'- {c["ref"]}: {c["text"][:150]}...' for c in chunks if word in words(c["text"])][:3])

jury = candidates[["word", "% of questions"]].copy()
for model in CONFIG["judge"]["jury"]:
    answers = []
    for n, w in enumerate(jury["word"], 1):
        answers.append(ask(model, PROMPT.format(examples=examples(w), word=w)))
        print(f"{model} {n}/{len(jury)}: {w} -> {answers[-1][:60]}")
    jury[model] = [label(a) for a in answers]
    jury[model + " reason"] = answers
models = CONFIG["judge"]["jury"]
jury["majority"] = np.where((jury[models] == "IGNORE").sum(axis=1) >= 2, "IGNORE", "KEEP")
JURY_IGNORE = set(jury.loc[jury["majority"] == "IGNORE", "word"])
print("The jury would ignore:", sorted(JURY_IGNORE))
print("Votes per model:", {m: jury[m].value_counts().to_dict() for m in models})
print("Models that gave the same label to every word (they do not tell words apart):",
      [m for m in models if jury[m].nunique() == 1] or "none")
jury[["word", "% of questions", *models, "majority"]]

**Comment:** The jury would ignore 11 function words (for example “the”, “of”, “and”, “shall”) and keeps the words our questions depend on, such as “ai”, “risk” and “high” (in 46–73% of the questions). Only Qwen3-4B told the words apart: phi4-mini answered KEEP and gemma3 answered IGNORE for almost every word, so the majority is the same as Qwen’s vote.

## 5. Does removing words help? Tested on my questions
Following the *burden of proof* rule, we compare three versions of BM25 on my questions **B-01 to B-06** (Annex III 3(b), planned for the `val` split):
- **none:** nothing removed (the default)
- **jury:** the words the jury marked IGNORE are removed
- **stopwords:** the scikit-learn stopword list is removed, but "not", "no", "nor" are kept

Scores, as in the instructor's slides (*"compute Recall@k / MRR on gold IDs"*): **Recall@5** = share of the gold provisions in the top 5; **MRR** = 1 / rank of the first correct chunk; **Precision@5** = share of the top 5 chunks that are gold provisions. A question with 4 gold provisions and 2 of them in the top 5 counts as Recall 0.5, not as a success: partial hits are counted as partial, and the unrelated chunks lower Precision.

**Rule:** a version replaces "none" only if it is better on Recall@5 or MRR and worse on neither. Six questions are few (one question changes Recall@5 by about 0.17), so this is a first check; it is repeated on all the team's `val` questions in KAN-99.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
REMOVE = {"none": set(), "jury": JURY_IGNORE, "stopwords": set(ENGLISH_STOP_WORDS) - {"not", "no", "nor"}}

def bm25_search(remove):
    index = BM25Okapi([[w for w in words(c["embed_text"]) if w not in remove] for c in chunks])
    return lambda q: [chunks[n]["chunk_id"] for n in np.argsort(-index.get_scores([w for w in words(q) if w not in remove]))]

test = pd.DataFrame([{"version": v, **score(bm25_search(words_out))} for v, words_out in REMOVE.items()]).set_index("version")
rule = ["Recall@5", "MRR"]                                   # the rule uses these two, as written above
better = lambda v: (test.loc[v, rule] >= test.loc["none", rule]).all() and (test.loc[v, rule] > test.loc["none", rule]).any()
CHOSEN = next((v for v in ["jury", "stopwords"] if better(v)), "none")
print("Chosen:", CHOSEN, "(write it in config/rag.yaml, bm25: remove)")
test

**Comment:** Removing the jury’s 11 words raised Recall@5 from 0.25 to 0.33 and MRR from 0.17 to 0.25, so by our rule BM25 now ignores them. The standard stopword list was close (0.29 and 0.26) but lower on Recall@5; with six questions this is a first check, repeated on the val split in KAN-99.

## 6. Dense, BM25 or hybrid?
- **Dense (E1, the baseline):** the chunks closest to the question vector, with the model in `config/rag.yaml` (bge-base, checked in `embedding.ipynb`).
- **BM25:** keyword search with the words chosen in step 5.
- **Hybrid (E2):** both lists joined with Reciprocal Rank Fusion: each chunk gets `1 / (60 + rank)` from each list, and the two are added. 60 is the value from the RRF paper (Cormack et al., 2009).

In [ ]:
from sentence_transformers import SentenceTransformer
R = CONFIG["retrieval"]
model = SentenceTransformer(CONFIG["embedding"]["model"], device="cpu")
vectors = model.encode([c["embed_text"] for c in chunks], normalize_embeddings=True)
INSTRUCTION = CONFIG["embedding"]["query_instruction"]
dense = lambda q: [chunks[n]["chunk_id"] for n in np.argsort(-(vectors @ model.encode([INSTRUCTION + q], normalize_embeddings=True)[0]))][:R["candidates"]]
keyword = lambda q: bm25_search(REMOVE[CHOSEN])(q)[:R["candidates"]]

def hybrid(question):
    rrf = Counter()
    for ranked in (dense(question), keyword(question)):
        for rank, cid in enumerate(ranked, 1):
            rrf[cid] += 1 / (R["rrf_k"] + rank)
    return [cid for cid, _ in rrf.most_common()]

pd.DataFrame([{"search": name, **score(fn)} for name, fn in [("dense (E1)", dense), ("BM25", keyword), ("hybrid (E2)", hybrid)]])

**Comment:** _(write 1–2 sentences after running)_

## 7. How many chunks should come back? (top-k)
Some of my questions need several provisions: B-05 needs eight Articles (8 to 15), but only 5 chunks come back, so it can never reach Recall 1. The design document (D12) plans to choose top-k from 3, 5 or 8. More chunks find more gold provisions but also bring more unrelated ones (lower Precision) and more text for the answer model to read.

In [ ]:
topk = pd.DataFrame([{"top-k": k, **dict(zip(["Recall", "MRR", "Precision"], score(hybrid, k).values()))} for k in [3, 5, 8]])
topk

**Comment:** _(write 1–2 sentences after running)_

## The decisions, written in `config/rag.yaml`
| Setting | Chosen | Step |
|---|---|---|
| `bm25.remove` | the LLM jury's 11 words | 5 |
| `retrieval.mode` | hybrid | 6 |
| `retrieval.top_k` | _(fill from step 7)_ | 7 |

## Next: KAN-99
- Repeat steps 5–8 on all the team's `val` questions (after KAN-53) and log them in W&B; compare bge-small and bge-base again (bge-small had the better Recall@5, bge-base the better MRR).
- The instructor's RAG metrics: Context Entities Recall (*"for fact-heavy text such as regulations"*) and, with the generated answers, Faithfulness.
- Questions that need several provisions (B-05; multi-hop questions like B-03 and B-06; the instructor's slides: *"the retriever has to find every document in the chain"*) can be Sprint 3 experiments: **query decomposition** (an LLM splits the question into one sub-question per provision) or **MMR** (Maximal Marginal Relevance, Carbonell & Goldstein 1998: the top chunks are chosen to be relevant **and** different from each other).